- Env 환경변수
- 기본 라이브러리

In [1]:
import os
import warnings

# Tokenizers 병렬 처리 경고 억제
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 경고 억제 (선택사항)
warnings.filterwarnings('ignore', category=UserWarning)

from dotenv import load_dotenv
load_dotenv()

True

In [24]:
import os
from glob import glob

# [실습 프로젝트] Naive RAG 구현 

- 각 단계별 지시사항에 따라 코드를 완성하세요. 
- 제시된 지시사항과 LangChain 문서를 참조하여 시스템을 구성합니다. 



## [구현] 투자 동향 보고서 RAG 

* 한경컨센서스 페이지에서 관련 파일 다운로드 및 업로드 하여 벡터 저장소 저장
https://consensus.hankyung.com/analysis/list?skinType=industry&sdate=2026-06-06&edate=2026-06-13&order_type=&now_page=2

* 문서를 RAG 벡터 저장소에 추가하기 위한 함수 구현
- 사용자가 질의하는 동안 벡터 저장소에 문서를 추가 할 수 있도록 함수 구현

1. 임베딩 모델 & 벡터 저장소 설정
2. PDF 파일 로드
3. 검색기 정의
4. RAG 프롬프트 구성
5. RAG 체인 구성
6. Gradio 스트리밍 구현
7. 저장소에 새문서 추가 로드
8. 추가 구현



In [3]:
# PDF 문서 추가 함수
# 파일을 지속적으로 추가하면서 테스트하기위해 함수로 구현

import os
import uuid
from typing import List, Optional
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def add_pdf_to_vectorstore(
    file_path: str,
    file_subject: str,
    file_author: str,
    vector_store,
    chunk_size: int = 1000,
    chunk_overlap: int = 200
) -> List[str]:
    """
    특정 PDF 파일을 로드, 청킹하여 벡터 저장소에 동적으로 추가합니다.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {file_path}")
        
    print(f"[{os.path.basename(file_path)}] 문서 로드 중...")
    
    # 1. 문서 로드
    loader = PyPDFLoader(file_path)
    docs = loader.load()
    
    # 2. 메타데이터 보강
    for doc in docs:
        doc.metadata["source_file"] = os.path.abspath(file_path)
        doc.metadata["subject"] = file_subject
        doc.metadata["author"] = file_author
        
    # 3. 텍스트 분할
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = text_splitter.split_documents(docs)
    
    # 4. 고유 ID 생성 및 문서 저장소 추가
    doc_ids = [str(uuid.uuid4()) for _ in range(len(chunks))]
    
    # 벡터 저장소에 추가
    added_ids = vector_store.add_documents(documents=chunks, ids=doc_ids)
    
    print(f"성공적으로 추가 완료! (생성된 청크 수: {len(added_ids)}개)")
    return added_ids

C:\Users\JSPark\AppData\Local\Temp\ipykernel_37016\51853389.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
e:\sw\dev\ai\modu_llm7\faq_bot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 벡터 저장소 설정
- HuggingFace에서 지원하는 BAAI/bge-m3 임베딩 모델을 사용하여 문서를 벡터화
- FAISS DB를 벡터 스토어로 사용 (IndexFlatL2 사용: 유클리드 거리)

In [4]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings  

# Hugging Face의 임베딩 모델 생성
# 힌트: HuggingFaceEmbeddings(model_name="BAAI/bge-m3") 사용
embeddings_model = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

# 임베딩 차원 확인
embedding = embeddings_model.embed_query("test")
print(f"임베딩 차원: {len(embedding)}")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 33804.81it/s]


임베딩 차원: 1024


In [5]:
# Ollama 임베딩 모델을 사용한 FAISS 벡터 저장소 생성
import faiss 
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

# FAISS 인덱스 초기화 (유클리드 거리 사용)
dim = 1024  # 임베딩 차원
faiss_index = faiss.IndexFlatL2(dim)  

# FAISS 벡터 저장소 생성
faiss_db = FAISS(
    embedding_function=embeddings_model,
    index=faiss_index,           # 벡터 검색을 위한 데이터 구조를 정의
    docstore=InMemoryDocstore(), # 문서 저장소 객체를 지정 - 문서의 원본 내용과 메타데이터를 보관
    index_to_docstore_id={},     # 인덱스와 문서 간의 연결을 관리 (매핑 딕셔너리)
)

# 저장된 문서의 갯수 확인
print(faiss_db.index.ntotal)

0


In [6]:
print(faiss_db.index.ntotal)

0


## 2. PDF 파일 로드

문서 목록
- './data/investment/6_3_유안타증권_20260612_market_554186000.pdf'
- './data/investment/냉정하게 볼 필요가 있는 금리 흐름_650022.pdf'
- './data/investment/보고서_20260611.pdf'

In [7]:
file1 = "./data/investment/6_3_유안타증권_20260612_market_554186000.pdf"
file2 = "./data/investment/냉정하게 볼 필요가 있는 금리 흐름_650022.pdf"
file3 = "./data/investment/보고서_20260611.pdf"


add_pdf_to_vectorstore(file1, "투자", "유안타증권", faiss_db)

[6_3_유안타증권_20260612_market_554186000.pdf] 문서 로드 중...
성공적으로 추가 완료! (생성된 청크 수: 21개)


['fcb7d60b-befd-417d-b587-a978fc63f26c',
 '4a04354d-7c46-4165-9735-87feeeb4a2f7',
 '7993184e-ef35-4dd7-a09c-849437245142',
 'f363b869-f571-4f47-9f36-327278790705',
 'e4298fbb-7bea-4a63-b148-1ef72a358769',
 '6b9bba98-217d-4de3-91ce-516a5506cf19',
 'fdc76289-a425-452c-91d5-73a4aa338200',
 '7fa47571-3853-47e6-910d-d219ecb8b44e',
 'd14a9483-0a47-448f-afed-a4ab074f7693',
 '8e18edd8-2923-4f38-a77e-b0d92b478d5a',
 '3387a0c5-e06a-4c21-b370-89a7298a523d',
 '98c4af5c-9c01-4636-8f4f-d09670780512',
 '3032c60b-d0fc-4c0a-9d68-d5ca48647c34',
 '54349f25-c769-40f5-a85d-7faaa1e492cb',
 'a7d9645d-0077-4c3f-b299-38a6510bd624',
 'ffed97fd-db16-4b7d-b3a0-e8a822d10271',
 'a0221017-0f9d-4830-a149-4aa975a3818e',
 '86ddcdfd-bb03-462c-a710-6277a24941c0',
 'b4e4e573-2247-4985-8611-25e687183c3a',
 '2e64a545-976f-4f93-9a0d-f3102160c497',
 '19d1220c-4e9b-4721-bd8f-921eb82d9a93']

In [8]:
# 추가 문서 로드
add_pdf_to_vectorstore(file2, "투자", "유안타증권", faiss_db)
add_pdf_to_vectorstore(file3, "투자", "유안타증권", faiss_db)

[냉정하게 볼 필요가 있는 금리 흐름_650022.pdf] 문서 로드 중...
성공적으로 추가 완료! (생성된 청크 수: 10개)
[보고서_20260611.pdf] 문서 로드 중...
성공적으로 추가 완료! (생성된 청크 수: 11개)


['21154b3d-8905-46ad-a205-1e5aa4e06238',
 '09cc7da8-90c7-42b0-9201-a3d3753b1814',
 '660e0c07-eaeb-45c4-b1a7-520edbb7da47',
 '3fa7c1b9-ce02-4749-afc6-bb4ce2f55511',
 '0056ed02-b386-4103-99dc-6fe7ca4d0bee',
 '3b357151-b076-4b79-b803-7262b2764b61',
 'e29522d6-4af4-4069-be09-d5f59827c22a',
 'd33b27d6-a6c4-466f-a3a8-03cb2b771922',
 '0afd64bc-1104-4d4d-88bb-62eee0ede15b',
 '93bef320-b946-4540-ad3f-3477af3cc200',
 'ae35090e-7717-4061-96aa-f59447ecf6fe']

## 3. 검색기 정의
- mmr 검색으로 상위 3개 문서 검색하는 Retriever 사용
- 다양성을 높이는 설정을 사용 

In [9]:
# mmr 검색기 생성
# 힌트: faiss_db.as_retriever(search_type='mmr', search_kwargs={'k': 3, 'fetch_k': 10, 'lambda_mult': 0.3})
# lambda_mult를 낮게 설정하여 다양성을 높임
faiss_mmr_retriever = faiss_db.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 3, 'fetch_k': 10, 'lambda_mult': 0.3}
    )

In [10]:
# 검색 테스트 
query = "금리란 무엇인가요??"
# 힌트: faiss_mmr_retriever.invoke(query) 사용
retrieved_docs = faiss_mmr_retriever.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"-{i}-\n{doc.page_content[:100]}...{doc.page_content[-100:]}")
    print("-" * 100)

쿼리: 금리란 무엇인가요??
검색 결과:
-1-
4 
 
금리는 연간 %로 표시된다.  
실질금리가 장기금리에 미치는 영향이 커진 점을 고려해, 경기 인식에 대한 부분이 금리에 미치
는 영향도 살펴봐야 한다. 경제지표 호전과 주...비용의 평가는 Repo금리 커브에 반영된다. 지난 3월 유가 급등과 경기지표 호전에 따
라 단기금리 커브는 스티프닝 압력이 이어졌다. 이에 따라 장기금리 상승 폭도 확대되는 양상이
----------------------------------------------------------------------------------------------------
-2-
Sector Report 
7 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Appendix 
 
 이 자료에 게재된 내용들은 본인의 의견을 정확하게 반영...임도 지지 않습니다. 또한, 본 자료는 당사 투자자에게만 제 공 되 는 자 료 로 당사의 동의 없이 본 자료를 무단으로 
복제 전송 인용 배포하는 행위는 법으로 금지되어 있습니다.
----------------------------------------------------------------------------------------------------
-3-
8
6월 15일(월) 6월 16일(화) 6월 17일(수) 6월 18일(목) 6월 19일(금)
경제지표 미국) 5월 산업생산 일본)BOJ 통화정책회의
미국) FOMC 회의
5월 소매...일 전망. 케빈워시의 첫 FOMC 기자회견 발언에 집중
6월 3주차주간일정
자료: 유안타증권리서치센터/ 주: 현지 시간 기준
주간 Market Calendar – 통화정책회의 주간
----------------------------------------------------------------------------------------------------


## 4. RAG 프롬프트 구성

- 작성 기준: 
    - LangChain의 ChatPromptTemplate 클래스 사용
    - 변수 처리는 {context}, {question} 형식 사용
    - 답변은 한글로 출력되도록 프롬프트 작성
    
- 아래 템플릿 코드를 기반으로 다음 내용을 참고하여 작성합니다. 

    1. 프롬프트 구성요소:
        - 작업 지침
        - 컨텍스트 영역
        - 질문 영역
        - 답변 형식 가이드

    2. 작업 지침:
        - 컨텍스트 기반 답변 원칙
        - 외부 지식 사용 제한
        - 불확실성 처리 방법
        - 답변 불가능한 경우의 처리 방법

    3. 답변 형식:
        - 핵심 답변 섹션
        - 근거 제시 섹션
        - 추가 설명 섹션 (필요시)

    4. 제약사항 반영:
        - 답변은 사실에 기반해야 함
        - 추측이나 가정을 최소화해야 함
        - 명확한 근거 제시가 필요함
        - 구조화된 형태로 작성되어야 함

In [11]:
# Prompt 템플릿 (여기에 작성하세요)
from langchain_core.prompts import ChatPromptTemplate

# 시스템 지침
system_message  = """당신은 주어진 컨텍스트(Context)만을 기반으로 정직하고 정확하게 답변하는 비서입니다.
[작업 지침]
1. 반드시 아래 제공된 컨텍스트(Context) 정보만을 바탕으로 질문에 답변하십시오.
2. 외부 지식이나 학습된 임의의 정보를 동원하여 사실을 왜곡하거나 임의로 유추하지 마십시오.
3. 답변이 불확실하거나 질문에 답하기 위한 정보가 부족한 경우, 가정을 배제하고 솔직하게 모른다고 답하십시오.
4. 컨텍스트에서 질문에 대한 답을 절대 찾을 수 없는 경우, 반드시 다음과 같이 답변하십시오: "제공된 컨텍스트에서 관련 정보를 찾을 수 없습니다."

[답변 형식 가이드]
반드시 다음 구조화된 형식을 엄격히 지켜 마크다운(Markdown) 형태로 답변을 작성해 주세요.

### 1. 핵심 답변
- 질문에 대한 핵심 결론과 직접적인 답변을 명확하고 간결하게 요약하여 작성합니다.
### 2. 근거 제시
- 답변의 근거가 된 컨텍스트 내의 구체적인 문장이나 핵심 단락을 그대로 인용하거나 명확하게 밝힙니다.
### 3. 추가 설명 (필요시)
- 핵심 답변을 보완하기 위해 컨텍스트에 포함되어 있는 유용한 추가 맥락이 있을 경우에만 작성합니다. (불필요한 경우 이 섹션은 생략 가능합니다.)

[제약사항]
- 답변은 100% 사실(Fact)에만 기반해야 합니다.
- 추측이나 가정을 절대 최소화하고 배제하십시오.
- 모든 답변에는 명확한 근거(근거 제시 섹션)가 포함되어야 합니다.
- 반드시 한국어로 자연스럽게 출력되도록 하십시오."""


# 사용자가 입력할 템플릿 ({context} 및 {question} 매핑)
user_message = """
[컨텍스트]
{context}

[질문]
{question}
"""

# ChatPromptTemplate 인스턴스 생성
prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_message),
    ("human", user_message)
])


# 템플릿 출력
prompt_template.pretty_print()

================================ System Message ================================

당신은 주어진 컨텍스트(Context)만을 기반으로 정직하고 정확하게 답변하는 비서입니다.
[작업 지침]
1. 반드시 아래 제공된 컨텍스트(Context) 정보만을 바탕으로 질문에 답변하십시오.
2. 외부 지식이나 학습된 임의의 정보를 동원하여 사실을 왜곡하거나 임의로 유추하지 마십시오.
3. 답변이 불확실하거나 질문에 답하기 위한 정보가 부족한 경우, 가정을 배제하고 솔직하게 모른다고 답하십시오.
4. 컨텍스트에서 질문에 대한 답을 절대 찾을 수 없는 경우, 반드시 다음과 같이 답변하십시오: "제공된 컨텍스트에서 관련 정보를 찾을 수 없습니다."

[답변 형식 가이드]
반드시 다음 구조화된 형식을 엄격히 지켜 마크다운(Markdown) 형태로 답변을 작성해 주세요.

### 1. 핵심 답변
- 질문에 대한 핵심 결론과 직접적인 답변을 명확하고 간결하게 요약하여 작성합니다.
### 2. 근거 제시
- 답변의 근거가 된 컨텍스트 내의 구체적인 문장이나 핵심 단락을 그대로 인용하거나 명확하게 밝힙니다.
### 3. 추가 설명 (필요시)
- 핵심 답변을 보완하기 위해 컨텍스트에 포함되어 있는 유용한 추가 맥락이 있을 경우에만 작성합니다. (불필요한 경우 이 섹션은 생략 가능합니다.)

[제약사항]
- 답변은 100% 사실(Fact)에만 기반해야 합니다.
- 추측이나 가정을 절대 최소화하고 배제하십시오.
- 모든 답변에는 명확한 근거(근거 제시 섹션)가 포함되어야 합니다.
- 반드시 한국어로 자연스럽게 출력되도록 하십시오.

================================ Human Message =================================


[컨텍스트]
{context}

[질문]
{question}



## 5. RAG 체인 구성

- LangChain의 LCEL 문법을 사용
- 검색 결과를 프롬프트의 'context'로 전달하고,
- 사용자가 입력한 질문을 그래도 프롬프트의 'question'에 전달
- LLM 설정:
    - ChatOpenAI 사용 ('gpt-4o-mini' 모델)
    - temperature: 답변의 일관성을 가져가는 설정값을 사용 
    - 기타 필요한 설정 
- 출력 파서: 문자열 부분만 출력되도록 구성

In [12]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# LLM 설정
# 힌트: ChatOpenAI(model='gpt-4o-mini', temperature=0) 사용
llm = ChatOpenAI(
    model="gpt-4.1-mini", 
    temperature=0
)

# 문서 포맷팅
def format_docs(docs):
    return "\n\n".join([f"{doc.page_content}" for doc in docs])

# RAG 체인 생성
# 힌트: {'context': faiss_mmr_retriever | format_docs, 'question': RunnablePassthrough()} | prompt | llm | StrOutputParser()
rag_chain = {'context': faiss_mmr_retriever | format_docs, 'question': RunnablePassthrough()} | prompt_template | llm | StrOutputParser()

# 체인 실행
#query = "정규화는 무엇인가요?"
query = "코스피 현황은 어떻게 되나요?"
output = rag_chain.invoke(query)

print(f"쿼리: {query}")
print("답변:")
print(output)

쿼리: 코스피 현황은 어떻게 되나요?
답변:
### 1. 핵심 답변
- 최근 주간(6/5~6/11) 코스피 지수는 10.1% 하락했으며, 대형주 차익실현 매물 출회 과정에서 반도체 소부장 중심으로 KOSDAQ이 상대적으로 강세를 보였습니다. 또한, KOSPI 지수는 지난 1년 사이 5천 포인트 상승하여 250%가 넘는 상승률을 기록했습니다.

### 2. 근거 제시
- "주간(6/5~6/11) KOSPI, KOSDAQ 각각 10.1%, 5.0% 하락."
- "대형주 대비 KOSDAQ 상대적 소외 지속됐으나 대형주 차익실현 매물 출회되는 과정에서 반도체 소부장 중심 KOSDAQ 상대적 강세"
- "KOSPI 지수는 지난 1년 사이에 5천 pt 상승하여, 250%가 넘는 상승율을 보였다."

### 3. 추가 설명
- 코스피 지수는 최근 금리 변동성과 지정학적 갈등 지속으로 인해 주간 큰 폭의 하락세를 보였으나, 장기적으로는 큰 폭의 상승을 기록한 상태입니다.  
- 또한, 금리와 경기지표 개선 속도에 따라 장기금리 변동성이 확대되고 있어 코스피 지수의 향후 움직임에 영향을 줄 수 있습니다.


## 6. Gradio 스트리밍 구현
- ChatInterface 사용
- `chain.stream()`으로 응답을 청크 단위로 스트리밍

In [13]:
import gradio as gr
from typing import Iterator

# 스트리밍 응답 생성 함수
def get_streaming_response(message: str, history) -> Iterator[str]:
    
    # RAG Chain 실행 및 스트리밍 응답 생성
    response = ""
    for chunk in rag_chain.stream(message):
        if isinstance(chunk, str):
            response += chunk
            yield response

# Gradio 인터페이스 설정
# 힌트: gr.ChatInterface(fn=get_streaming_response, title="RAG 기반 질의응답 시스템", description="...", examples=[...])
demo = gr.ChatInterface(
    fn=get_streaming_response, 
    title="RAG 기반 질의응답 시스템", 
    description="사용자 질의에 대한 최신 정보를 답변합니다.", 
    examples=["최신 상승 주식은 무엇인가요?",
    "미국 이란 갈등은 어떻게 되고있나요?", 
    "마지막 KOSPI 지수는 어떻게 되었나요?",
    ])

# 실행
demo.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## 7. 저장소에 새문서 추가 로드

- 새 파일을 업로드 후 챗봇이 몰라서 답변을 못한 내용을 다시 gradio에서 질문

In [14]:
print(f"현재 총 FAISS 문서 개수: {faiss_db.index.ntotal}")

# 다시 gradio에서 검색 - 동적 검색 확인 완료

현재 총 FAISS 문서 개수: 42


In [15]:
# 추가 PDF 문서 로드
new_file = "./data/investment/뷰티산업_649930.pdf"

add_pdf_to_vectorstore(new_file, "투자", "LS증권", faiss_db)

[뷰티산업_649930.pdf] 문서 로드 중...
성공적으로 추가 완료! (생성된 청크 수: 116개)


['d6c9e892-b86c-48c5-a700-802b0df89665',
 '4d6c89ab-e27a-4a3f-ab38-7ef0ed500e21',
 '9e5a2114-08a7-4166-9c5a-38dfd8e7c7a3',
 '91508c05-c43c-461a-a69c-2bc29535ebd1',
 '0b247442-be91-492a-9324-21d67e229ac8',
 'a7e67fc9-b373-484c-bb41-dbee9b161f42',
 'b037e38c-5276-41cf-a05c-d16142a9508a',
 '26a0e189-03ba-4de9-8816-a7943f9a4913',
 'fe7511a3-8faa-4096-9223-a127db43ac2d',
 '78675b58-34f7-477d-949a-b04412fac690',
 '22c1a7ea-7816-4061-a16b-43c3645a3fc3',
 '95db4993-9745-47ad-ab84-bee99cbdd5f9',
 '79dd9aa7-1062-4744-950d-bfbd09f311ef',
 '6ad08328-426e-486b-9908-87dad7431597',
 '6d6aa4f7-5b0a-492a-a025-402332f4bf78',
 'e0781a7f-9b8e-497a-9dd6-49b8e23034fa',
 'e17fa5a6-9555-44e5-a23d-3283a8564583',
 '615e0cdd-05c5-4a7b-bae5-d6973b955e35',
 'b5242fc7-0dbe-4d2e-b15f-92c158bdb076',
 '16aafbd7-66c5-41fd-9b13-31ecac07d103',
 '4b9d8a0e-5e4f-4f31-ab8b-b2898a4adfe4',
 '0432c6ae-5f7d-48f4-b851-3410cb988598',
 'cb41314c-3711-48d2-8b6b-637348186813',
 '9046d9e7-407c-4693-947a-57a6478cc198',
 '833b07c9-0c5d-

In [ ]:
# demo 실행 종료
demo.close()

## 8. 추가 구현

- 저장소의 데이터가 많아질 경우 메타 데이터를 분류하여, 사용자의 질의에 따라 특정 메타 데이터 기준으로 분류하여 조회하도록 구현 


In [16]:
# 1. FAISS 내부의 InMemoryDocstore 딕셔너리에 저장된 모든 문서 객체 가져오기
all_documents = list(faiss_db.docstore._dict.values())

# 2. 각 문서의 metadata에서 'category' 키 값 추출하여 중복 제거

unique_categories = set()

for doc in all_documents:
    meta = doc.metadata
    if meta and "subject" in meta:
        unique_categories.add(meta["subject"])

# 3. 리스트 형태로 변환
ALLOWED_CATEGORIES = list(unique_categories)
print(f"FAISS DB에서 가져온 카테고리 목록: {ALLOWED_CATEGORIES}")

FAISS DB에서 가져온 카테고리 목록: ['투자']


In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_chroma import Chroma

# --------------------------------------------------
# 분석을 위한 Pydantic 스키마 정의
# --------------------------------------------------
class QueryClassifier(BaseModel):
    # LLM이 사전에 약속된 카테고리 목록 내에서만 매핑하도록 설명(Description)에 가이드 제공
    subject: Optional[str] = Field(
        default=None,
        description=f"질문의 주제와 가장 일치하는 항목을 선택하세요. 대상 목록: {ALLOWED_CATEGORIES}. 만약 일치하는 항목이 전혀 없거나 일반적인 대화라면 None을 반환하세요."
    )
    refined_query: str = Field(
        description="벡터 데이터베이스 검색을 위해 핵심 키워드 위주로 정리된 질의문."
    )
# --------------------------------------------------
# 3. 분류 모델 및 프롬프트 생성
# --------------------------------------------------

llm2 = ChatOpenAI(model="gpt-4o-mini", temperature=0)

structured_llm = llm2.with_structured_output(QueryClassifier)

system_prompt = f"""당신은 사용자 질문을 분석하여 사전 정의된 카테고리 목록에 매핑하는 분류기입니다.
[사전 정의된 카테고리 목록]
{ALLOWED_CATEGORIES}

[분류 규칙]
1. 사용자의 질문 주제가 목록 내의 어떤 카테고리와 '의미상' 유사하거나 연관이 깊은지 판단하십시오.
2. 반드시 제공된 목록 내에 있는 정확한 문자열 형태로만 카테고리를 선택해야 합니다. 목록 외의 새로운 단어를 지어내지 마십시오.
3. 질문의 주제가 목록 내의 어떤 카테고리와도 무관하다면 category에 None을 지정하십시오.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "질문: {question}")
])

classifier_chain = prompt | structured_llm

# --------------------------------------------------
# 4. 연합 필터링 검색 실행 함수
# --------------------------------------------------
def retrieve_with_aligned_metadata(question: str, faiss_db: FAISS, k: int = 3):
    # Step 1: 사용자 질문 분류 수행
    result: QueryClassifier = classifier_chain.invoke({"question": question})
    
    print(f"\n[입력 질문]: '{question}'")
    print(f"🎯 매핑된 메타데이터: {result.subject}")
    print(f"🔍 검색 쿼리: '{result.refined_query}'")
    
    # Step 2: 메타데이터 필터 조건 설정
    search_kwargs = {"k": k, 'fetch_k': 10, 'lambda_mult' : 0.3}
    
    # 매핑된 카테고리가 존재하고 사전 정의된 목록에 있는 경우 필터 구성
    if result.subject and result.subject in ALLOWED_CATEGORIES:
        search_kwargs["filter"] = {
            "subject": result.subject  # 사전 약속된 메타데이터 값으로 필터 적용
        }
        print(f"✅ 필터 적용 : {{'subject': '{result.subject}'}}")
    else:
        print("ℹ️ 적용 가능한 메타데이터 카테고리가 없어 필터 없이 전체 검색을 수행합니다.")
        
    # Step 3: 검색 실행
    #retriever = chroma_db.as_retriever(search_kwargs=search_kwargs)

    retriever = faiss_db.as_retriever(
    search_type='mmr',
    search_kwargs=search_kwargs
    )
    
    retrieved_docs = retriever.invoke(result.refined_query)
    
    return retrieved_docs

In [23]:
# 수정된 카테고리로 필터링해서 검색하기
query = "금리 혹은 마켓 분석에 대한 질문"\

# 'filter' 파라미터에 수정했던 키와 값을 지정합니다.
retrieved_docs = retrieve_with_aligned_metadata(query, faiss_db)

print(f"검색된 문서 개수: {len(retrieved_docs)}")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n[{i}번째 문서]")
    print(f"본문 내용: {doc.page_content[:150]}...")
    print(f"메타데이터: {doc.metadata}")


[입력 질문]: '금리 혹은 마켓 분석에 대한 질문'
🎯 매핑된 메타데이터: 투자
🔍 검색 쿼리: '금리와 마켓 분석에 대한 질문'
✅ 필터 적용 적용: {'subject': '투자'}
검색된 문서 개수: 3

[1번째 문서]
본문 내용: 4 
 
금리는 연간 %로 표시된다.  
실질금리가 장기금리에 미치는 영향이 커진 점을 고려해, 경기 인식에 대한 부분이 금리에 미치
는 영향도 살펴봐야 한다. 경제지표 호전과 주가지수 상승으로 실질 및 명목금리 상승에 영향을 
미친 점이 있다. KOSPI 지수는 지난...
메타데이터: {'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-06-12T06:53:10+09:00', 'moddate': '2026-06-12T06:53:10+09:00', 'source': './data/investment/냉정하게 볼 필요가 있는 금리 흐름_650022.pdf', 'total_pages': 7, 'page': 3, 'page_label': '4', 'source_file': 'e:\\sw\\dev\\ai\\modu_llm7\\faq_bot\\data\\investment\\냉정하게 볼 필요가 있는 금리 흐름_650022.pdf', 'subject': '투자', 'author': '유안타증권'}

[2번째 문서]
본문 내용: Industry In depth / 화장품 / 2026. 6. 9  
LS Securities Research 56 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Compliance Notice 
본 자료에 기재된 내용들...
메타데이터: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-06-08T18:06:11+09:00', 'a